# 학교 계정 · Dunnhumby GCN seed51 · 8모형

학교·개인 Drive 파일명 검색에서 GCN seed51 결과/체크포인트 미발견(2026-10-24). 검색으로 모든 미게시 실행까지 부재를 증명하지는 못합니다. seed50 8모형 완료 후 같은 GPU에서 실행하세요.

seed50과 동일한 모델·입력·300epoch·batch8192·학습률·규제·uniform 음성표본을 유지합니다. DAY1~704 학습, DAY705~711 신규상품 추천 test, MIN_ITEM_INTER=1, validation/holdout/중간평가/조기종료 없음. M1/M2/M3/M4/M5_A~D 각 마지막점만 평가합니다. seed만51로 바뀌며 출력은 분리합니다.

### 학교 Drive 저장 보호
- 원본 ZIP을 재사용하며 압축 해제는 Colab 임시 디스크에서만 합니다.
- 이번 GCN 예산0.75GB. 학교 계정 소유 파일·휴지통·보존 버전을 합산하고 **다른 두 실행의 교체 저장4GB + 비파일/동기화 여유1GB**를 남깁니다.
- 용량 조회는60초 캐시하고 그동안 GCN 저장 요청량을 보수적으로 차감합니다. API 일시 제한에는 단계적으로 재시도합니다.
- 진행 중 최신·직전 체크포인트를 보존합니다. 완료 모형도 로컬 백업 검증 전에는 직전본을 유지합니다. 동일 실행의 검증된 더 오래된 휴지통 사본만 정리합니다.
- 완료 결과가 계속 늘면 안전 중단할 수 있습니다. **세 실행 전체 완료까지10GB를 보장하는 정책은 아닙니다.** 다른 두 런타임을 이 노트북이 통제하지 않습니다. Mac에 완료 가중치·결과·설정·이력을 내려받아 해시/내용 검증 후 학교 사본을 정리하는 별도 절차가 필요합니다. `/content`는 영구 백업이 아닙니다.
- 기존 seed50이 미완료이거나 다른 seed51 출력이 발견되면 실행을 차단합니다. 같은 출력의 동시 실행은 금지합니다.

학교 계정 L4 GPU로 위에서부터 실행하세요. 실제 GPU 학습은 아직 실행하지 않았습니다. 기존 세 런타임과 파일은 이 준비 작업에서 변경하지 않았습니다.

재개 시 용량 검사 전에 기존 실행의 오래된 휴지통 체크포인트를 내용 검증 후 정리합니다. 최신·직전본과 완료 결과는 유지합니다. 정리 후에도 공간이 부족하면 안전 중단하며, 이미 완료된 모형은 재학습하지 않습니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 L4 GPU로 설정하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU: {GPU}. L4로 변경 후 실행하세요.'
SOURCE_COMMIT = '05237af3c3c13883f6e4d59cce1d1b641af5caed'
REPO = Path('/content/clv-dh-gcn-s51-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 학교 계정·용량 확인
Drive 연결과 동일한 학교 계정으로 인증하세요.

In [ ]:
import hashlib, urllib.request, types
url = 'https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/62a19e044753b8259674903c7cd29c580798c535/clv_drive_checkpoint_retention.py'
with urllib.request.urlopen(url, timeout=60) as response:
    retention_source = response.read()
assert hashlib.sha256(retention_source).hexdigest() == '32a69a7ba5e3f3d7c0f9ca3b0e0731d3075f8b547069afa4df3308f0bf873e98'
retention = types.ModuleType('verified_drive_retention')
exec(compile(retention_source, '<verified-drive-retention>', 'exec'), retention.__dict__)
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
(credentials, _) = google.auth.default()
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
OTHER_RUN_RESERVE_GB = 5.0  # 다른 두 실행의 교체 저장 4GB + 비파일/동기화 여유 1GB

def query_space(required_bytes):
    info = retention._execute(drive_api.about().get(fields='user(emailAddress),storageQuota'))
    if info['user']['emailAddress'] != 'jungunb@u.sogang.ac.kr':
        raise RuntimeError('학교 계정으로 인증하세요.')
    (used, token, seen) = (0, None, set())
    while True:
        page = retention._execute(drive_api.files().list(q="'me' in owners", spaces='drive', corpora='user', pageSize=1000, pageToken=token, fields='nextPageToken,incompleteSearch,files(id,mimeType,quotaBytesUsed)'))
        if page.get('incompleteSearch'):
            raise RuntimeError('개인 파일 용량 조회가 불완전합니다. 저장을 중단합니다.')
        for f in page.get('files', []):
            if f['id'] in seen:
                raise RuntimeError('파일 목록 중복: 동시 변경 가능성이 있어 재조회가 필요합니다.')
            seen.add(f['id'])
            amount = f.get('quotaBytesUsed')
            if amount is None:
                if f['mimeType'] in ('application/vnd.google-apps.folder', 'application/vnd.google-apps.shortcut'):
                    continue
                raise RuntimeError('일부 파일의 용량을 확인하지 못했습니다. 저장을 중단합니다.')
            if int(amount) < 0:
                raise RuntimeError('유효하지 않은 파일 사용량')
            used += int(amount)
        token = page.get('nextPageToken')
        if not token:
            break
    available = 10 * 10 ** 9 - used
    needed = required_bytes + int(OTHER_RUN_RESERVE_GB * 10 ** 9)
    quota = info.get('storageQuota', {})
    if 'limit' in quota and 'usage' in quota:
        pool_available = int(quota['limit']) - int(quota['usage'])
        if pool_available < needed:
            raise RuntimeError('학교 조직 전체 저장공간의 여유가 부족합니다.')
    if available < needed:
        raise RuntimeError(f'개인 Drive 파일 기준 여유 {available / 1000000000.0:.2f}GB, 필요 {needed / 1000000000.0:.2f}GB. 기존 파일은 보존하며 중단합니다.')
    return (info['user']['emailAddress'], available)
_quota_cache = None
_quota_at = 0.0
_quota_charged = 0

def check_space(required_bytes):
    global _quota_cache, _quota_at, _quota_charged
    import time
    if _quota_cache is None or time.monotonic() - _quota_at >= 60:
        _quota_cache = query_space(required_bytes)
        _quota_at = time.monotonic()
        _quota_charged = 0
    account, available = _quota_cache
    if available - _quota_charged < required_bytes + int(OTHER_RUN_RESERVE_GB * 10**9):
        raise RuntimeError('다른 두 실행의 저장 여유 부족: 기존 체크포인트를 유지하고 중단합니다. 완료 결과를 검증 백업한 뒤 재개하세요.')
    _quota_charged += required_bytes  # cached quota never treats an intervening write as free
    return account, available - _quota_charged

# 용량 검사보다 먼저 기존 실행의 검증 가능한 오래된 사본을 정리합니다.
# 이 단계는 학습/평가/prepare를 실행하지 않으며 최신·직전본을 유지합니다.
_existing_run = Path('/content/drive/MyDrive/논문/data/results_v3_dunnhumby_gcn_final8_seed51_school_v1/3a93e11f5f4b')
_owner = retention._execute(drive_api.about().get(fields='user(emailAddress)'))['user']['emailAddress']
assert _owner == 'jungunb@u.sogang.ac.kr', '학교 계정으로 인증하세요.'
_checkpoints = list((_existing_run / 'progress' / 'resume').glob('*_latest.pt'))
# 모든 로컬 신원과 복구 사본을 먼저 확인한 뒤 정리를 시작합니다.
for _path in _checkpoints:
    _identity, _epoch = retention._checkpoint_info(_path)
    assert _identity.get('seed') == 51 and _identity.get('config_hash') == '3a93e11f5f4b', '기존 실행 신원 불일치: 정리하지 않습니다.'
    assert _identity.get('source_revision') == SOURCE_COMMIT, '기존 학습 코드 불일치: 정리하지 않습니다.'
    for _recovered in _path.parent.glob(_path.name.replace('_latest.pt', '_recovered_ep*.pt')):
        _rid, _rep = retention._checkpoint_info(_recovered)
        assert _rid == _identity and _rep <= _epoch, '더 최신 복구 사본이 있습니다. 삭제/재개하지 않습니다.'
for _path in _checkpoints:
    retention.prune(drive_api, _path, keep_previous=1)
print('기존 GCN 체크포인트 확인:', len(_checkpoints), '개 / 최신·직전·완료 결과 유지')
print('개인 소유 Drive 파일·휴지통·보존 버전 합산. Gmail/Photos는 미포함이며 총5GB 여유를 남깁니다.')
(account, available) = check_space(750 * 10 ** 6)
print('용량 확인 계정:', account, '| 개인 Drive 파일 기준 여유 추정 GB:', round(available / 1000000000.0, 2))
import uuid, time
probe = Path('/content/drive/MyDrive') / ('.clv_gcn_account_' + uuid.uuid4().hex)
probe.write_text('GCN account verification')
try:
    found = []
    for attempt in range(10):
        found = retention._execute(drive_api.files().list(q=f"name = '{probe.name}' and trashed = false and 'me' in owners", fields='files(id)', pageSize=10)).get('files', [])
        if found:
            break
        time.sleep(2)
    assert len(found) == 1, 'Drive 마운트와 용량 확인 계정이 다르거나 동기화가 지연되었습니다. 같은 계정으로 연결 후 재시도하세요.'
finally:
    probe.unlink(missing_ok=True)


## 입력 검증·준비
기존 seed49 참조는 고정 입력 검증에만 사용합니다. 이번 학습·저장·평가는 seed51입니다.

In [ ]:
import hashlib, shutil, zipfile
import clv_dh_gcn_final8_seed51 as screen
assert Path(screen.__file__).resolve().parent == REPO.resolve(), '이 런타임을 재시작하고 위에서부터 실행하세요.'
BUNDLE = Path('/content/drive/MyDrive/dh_gcn_school_inputs_v1.zip')
EXPECTED = '22cbe0f0e6e0cb772e1391533e3fd6526e2fa42a0c5e2c5b4b2807c376b62828'
def sha(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(8*1024**2), b''):h.update(block)
    return h.hexdigest()
assert BUNDLE.is_file(), '학교 Drive 최상위의 준비된 dh_gcn_school_inputs_v1.zip을 확인하세요.'
assert sha(BUNDLE) == EXPECTED, '입력 ZIP 해시 불일치'
LOCAL = Path('/content/dh_gcn_school_inputs_v1')
LOCAL.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    manifest = json.loads(z.read('manifest.json'))
    assert set(manifest) == {'transaction_data.csv','product.csv','m2_reference.zip','m4_reference.zip'}
    for name, expected in manifest.items():
        target = LOCAL / name
        if not target.is_file() or sha(target) != expected:
            temporary = target.with_suffix('.partial')
            with z.open(name) as source, temporary.open('wb') as sink:shutil.copyfileobj(source,sink)
            assert sha(temporary) == expected, '원본 복사 불일치: ' + name
            temporary.replace(target)
screen.base.v3.SCHEMA['dunnhumby'].update(tx_path=str(LOCAL/'transaction_data.csv'),item_meta_path=str(LOCAL/'product.csv'))
cfg = screen.configure(out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_gcn_final8_seed51_school_v1',
                       m2_reference=str(LOCAL/'m2_reference.zip'),m4_reference=str(LOCAL/'m4_reference.zip'))
def storage_guard(required_bytes):
    check_space(required_bytes)
    root = Path(cfg.out_dir)
    used = sum(p.stat().st_size for p in root.rglob('*') if p.is_file()) if root.exists() else 0
    if used + required_bytes + BUNDLE.stat().st_size > 750 * 10**6:
        raise RuntimeError('이번 GCN 저장 예산0.75GB 초과 예방: 기존 파일을 보존하고 중단합니다.')
screen.STORAGE_GUARD = storage_guard
storage_guard(150 * 10**6)
# seed50 완료 후 GPU를 넘겨받습니다. 미완료 실행과 새 seed를 겹치지 않습니다.
PREVIOUS = Path('/content/drive/MyDrive/논문/data/results_v3_dunnhumby_gcn_final8_seed50_school_v1/865a1c2465a9')
for label in screen.ORDER:
    arm = PREVIOUS / 'arms' / f'gcn_{label.lower()}_lastweek_s50.json'
    assert arm.is_file(), f'seed50 {label} 완료 결과가 없습니다. seed50을 먼저 완료하세요.'
    row = json.loads(arm.read_text())
    assert row.get('seed') == 50 and row.get('epochs') == 300 and row.get('model') == label, 'seed50 완료 신원 불일치'
    assert row.get('identity', {}).get('config_hash') == '865a1c2465a9', 'seed50 실행 신원 불일치'
# 동일 seed의 다른 실행 폴더가 있으면 자동으로 새 학습을 만들지 않습니다.
parent = Path(cfg.out_dir).parent
conflicts = [p for p in parent.glob('*gcn*seed51*') if p.resolve() != Path(cfg.out_dir).resolve()]
assert not conflicts, f'기존 seed51 폴더를 먼저 확인하세요: {conflicts}'
prepared = screen.prepare(cfg)
other_protocols = [p for p in Path(cfg.out_dir).glob('*/protocol.json') if p.parent != prepared['run_dir']]
assert not other_protocols, f'다른 seed51 설정/코드 결과가 있습니다. 기존 실행을 확인하세요: {other_protocols}' 
print('원본 검증 완료 / seed51 / 8모형 / 각300epoch:', prepared['run_dir'])


## 2. GCN 8개 모형 순차 학습
모형당 시작3개 epoch 및 이후5epoch마다 손실·전체 학습 시간·Drive 저장 시간을 표시합니다. 처음5배치가 끝나면 입력/음성추출과 학습계산의 배치당 표본 시간도 먼저 표시하므로 첫epoch가 끝나기 전에도 확인할 수 있습니다.

매epoch checkpoint를 Drive에 저장하고, 각 모형300epoch 완료 시 전체142지표를 평가·저장한 뒤 다음 모형으로 넘어갑니다. **로그 빈도와 무관하게 Drive 저장은 매epoch 실행**됩니다. 학습 중 다른 셀을 실행하거나 런타임을 바꾸지 마세요.

각 시간은 다음과 같습니다.
- 입력·음성추출: 같은 순서의 uniform 음성표본 생성과 배치 자료 준비/전송.
- 학습계산: 그래프 전파·BPR 손실·역전파·Adam 업데이트.
- 학습 전체: epoch의 입력·계산·로그/진행기록을 포함한 벽시계 시간.
- Drive 저장완료: 로컬 직렬화·복사/해시·Drive 저장/메타데이터 전체 시간.


## 체크포인트 보존 설치
최신본의 크기·MD5·소유자·폴더 검증 후, 같은 실행의 최신본과 직전본을 남기고 더 오래된 명시적 휴지통 사본만 정리합니다. 일반 휴지통 전체를 비우지 않습니다. 확인 실패 시 삭제 없이 중단합니다.


In [ ]:
# 학습 셀이 멈춘 상태에서, 입력 준비 완료 후 한 번 실행하세요.
# 이 실행의 검증된 오래된 체크포인트 휴지통 사본만 영구 삭제합니다.
assert 'screen' in globals() and 'prepared' in globals(), '먼저 기존 입력 준비 셀을 실행하세요.'
if 'drive_api' not in globals():
    from google.colab import auth
    auth.authenticate_user()  # 반드시 학교 계정 선택
    import google.auth
    from googleapiclient.discovery import build
    credentials, _ = google.auth.default()
    drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
import hashlib, importlib.util, urllib.request
from pathlib import Path
retention_url = 'https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/62a19e044753b8259674903c7cd29c580798c535/clv_drive_checkpoint_retention.py'
with urllib.request.urlopen(retention_url, timeout=60) as response:
    retention_code = response.read()
assert hashlib.sha256(retention_code).hexdigest() == '32a69a7ba5e3f3d7c0f9ca3b0e0731d3075f8b547069afa4df3308f0bf873e98', '보존 도우미 해시 불일치'
retention_path = Path('/content/clv_drive_checkpoint_retention_6f20874.py')
retention_path.write_bytes(retention_code)
retention_spec = importlib.util.spec_from_file_location('clv_drive_checkpoint_retention_patch', retention_path)
retention = importlib.util.module_from_spec(retention_spec)
retention_spec.loader.exec_module(retention)
# Drive API 반영 지연을 최대 약5분 기다린 뒤 안전 중단합니다.
retention.prune.__kwdefaults__['retries'] = 30
# 로컬 백업 검증 전에는 완료 모형도 직전본을 삭제하지 않습니다.
_original_prune = retention.prune
def preserve_previous(*args, **kwargs):
    kwargs['keep_previous'] = 1
    return _original_prune(*args, **kwargs)
retention.prune = preserve_previous
retention.install(screen, drive_api, prepared)


In [ ]:
assert getattr(getattr(screen, 'original', screen).SafeProgressStore, '_retention_installed', False), '보존 셀을 먼저 실행하세요. 정리 기능 없이 학습하지 않습니다.'
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 시간 기록 확인 — 학습·평가 없음
아래 셀은 저장된 시간 CSV만 읽습니다. 학습 셀이 끝났거나 중단된 뒤 실행하세요. 마지막10개 epoch를 보여주며 전체 기록은 결과 ZIP의 `timing/`에 포함됩니다. 입력 준비 총 시간은 `preparation_timing.json`에 있습니다. 표본 시간은 첫3개 epoch만 있으며, 나머지 빈칸은 미측정입니다.

In [ ]:
import pandas as pd
from IPython.display import display
timing_files = sorted((prepared['run_dir'] / 'progress' / 'timing').glob('*.csv'))
if timing_files:
    timing = pd.concat([pd.read_csv(p) for p in timing_files], ignore_index=True)
    columns = ['model_id', 'epoch', 'train_seconds', 'save_seconds',
               'sample_input_seconds_per_batch', 'sample_compute_seconds_per_batch',
               'checkpoint_local_seconds', 'checkpoint_drive_seconds', 'checkpoint_metadata_seconds']
    display(timing[columns].tail(10))
    print('단위: 초. 표본 계측은 추가 동기화 비용을 포함합니다. test 성능을 측정하지 않습니다.')
else:
    print('아직 완료 epoch의 시간 기록이 없습니다. 초기 배치 시간은 학습 출력에서 확인하세요.')

## 3. 저장된 완료 모형만 비교 — 학습·재평가 없음
8모형 절대지표와 **같은 GCN M1 대비** 변화율입니다. 미완료 모형은 빈칸이며0점이 아닙니다. 전체142지표·모든 모형 간 비교·개입 진단은 CSV에 보관됩니다. 단일 시드이므로 표준편차/신뢰구간을 만들지 않습니다.

세션이 끊겼다면1번 준비 셀까지 실행한 다음 이 셀로 바로 와도 됩니다. 전체 결과를 받으려면2번 학습 셀을 이어서 실행하세요.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(columns=screen.ORDER))
against_m1 = result['comparison'].query("reference == 'M1'")
if not against_m1.empty:
    display(against_m1.pivot(index='metric', columns='model', values='relative_change_pct')
            .reindex(index=metrics, columns=screen.ORDER[1:]))
    print('GCN M1 대비 상대변화(%). 모든 비교·불리한 지표도 CSV에 보존합니다.')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
print('seed51 1시드 — 평균/표준편차·유의성·CLV 배정 귀속 판단 아님')

## 4. 결과 ZIP 다운로드
완료 모형의 원본 결과·1~300epoch 학습 이력·전체 비교·설정·입력 검증 기록을 받습니다. 8개 미완료 시 파일명에 `_partial`을 붙입니다. 재개용 대용량 checkpoint는 ZIP에 넣지 않고 **Drive에 그대로 유지**합니다.

In [ ]:
from google.colab import files
import shutil
output = screen.export_zip(prepared, '/content/dh_gcn_final8_seed51_results.zip')
src = Path(output)
backup = prepared['run_dir'] / src.name
storage_guard(src.stat().st_size)
shutil.copy2(src, backup)
assert backup.stat().st_size == src.stat().st_size
print('Drive 보관:', backup, '| ZIP 크기(MB):', round(src.stat().st_size / 1024**2, 3))
print('다운로드가 멈추면 위 Drive 경로에서 직접 다운로드하세요. checkpoint는 Drive에 유지됩니다.')
files.download(str(src))